# LeetCode 115 — Distinct Subsequences (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def numDistinct(self, s: str, t: str) -> int:

```

**Mục tiêu sau khi học xong:**

1. Phân biệt rõ **subsequence (dãy con)** với **substring (chuỗi con liên tiếp)**, và hiểu đúng chữ "distinct" trong đề: đếm **các cách chọn vị trí khác nhau**, không phải đếm các chuỗi khác nhau.
2. Viết **Cách 1 — brute force** bằng `itertools.combinations` (sinh mọi bộ chỉ số) và hiểu vì sao nó bùng nổ.
3. Học **đệ quy (recursion)** từ căn bản, viết **Cách 2 — đệ quy "chọn / không chọn" `s[i]`**.
4. Học **memoization** rồi viết **Cách 3 — đệ quy + memo (top-down DP)**, biết cái bẫy **giới hạn độ sâu đệ quy** của Python.
5. Viết **Cách 4 — bảng DP `dp[i][j]` bottom-up**, tự tay điền bảng cho `"rabbbit"` / `"rabbit"`.
6. Viết **Cách 5 — DP 1 chiều duyệt `j` ngược** và giải thích được **vì sao bắt buộc phải duyệt ngược**.
7. Nộp được bản `class Solution`, kiểm thử chéo 5 cách với nhau, và tự giải các biến thể (liệt kê mọi cách chọn, Distinct Subsequences II).

## 1. Đề bài

Cho hai chuỗi `s` và `t`, hãy trả về **số lượng subsequence phân biệt (distinct subsequences)** của `s` mà bằng `t`.

Các test case được sinh sao cho đáp án vừa với **số nguyên có dấu 32-bit** (32-bit signed integer).

**Ví dụ 1**

```text
Input: s = "rabbbit", t = "rabbit"
Output: 3
Giải thích:
Như minh họa bên dưới, có 3 cách để tạo ra "rabbit" từ s.
rabbbit
rabbbit
rabbbit
```

(Trên LeetCode, mỗi dòng `rabbbit` có gạch chân các ký tự được chọn; cell code ngay dưới sẽ vẽ lại 3 cách đó.)

**Ví dụ 2**

```text
Input: s = "babgbag", t = "bag"
Output: 5
Giải thích:
Như minh họa bên dưới, có 5 cách để tạo ra "bag" từ s.
babgbag
babgbag
babgbag
babgbag
babgbag
```

**Constraints (ràng buộc):**

- `1 <= s.length, t.length <= 1000`
- `s` và `t` chỉ gồm các chữ cái tiếng Anh (English letters).

> Hai con số cần ghi nhớ: độ dài tới **1000** (nên thuật toán `O(n·m)` = 10⁶ phép tính là ổn, còn thử mọi tập con là không thể), và đáp án **vừa 32-bit** (nhưng các giá trị *trung gian* có thể lớn hơn — ta sẽ quay lại chuyện này ở phần DP).

In [1]:
def show_way(s: str, chosen: tuple) -> str:
    # In s, bên dưới đánh dấu ^ ở các vị trí được chọn
    marks = "".join("^" if i in chosen else " " for i in range(len(s)))
    picked = "".join(s[i] for i in chosen)
    return f"{s}\n{marks}   chỉ số {chosen} -> {picked!r}"


s, t = "rabbbit", "rabbit"
print("Chỉ số:", " ".join(f"{i}:{ch}" for i, ch in enumerate(s)))
# 3 cách của Ví dụ 1: chỉ khác nhau ở việc bỏ chữ 'b' nào trong 3 chữ 'b' (chỉ số 2, 3, 4)
ways_example1 = [(0, 1, 2, 3, 5, 6), (0, 1, 2, 4, 5, 6), (0, 1, 3, 4, 5, 6)]
for w in ways_example1:
    print(show_way(s, w))

Chỉ số: 0:r 1:a 2:b 3:b 4:b 5:i 6:t
rabbbit
^^^^ ^^   chỉ số (0, 1, 2, 3, 5, 6) -> 'rabbit'
rabbbit
^^^ ^^^   chỉ số (0, 1, 2, 4, 5, 6) -> 'rabbit'
rabbbit
^^ ^^^^   chỉ số (0, 1, 3, 4, 5, 6) -> 'rabbit'


### Đọc kỹ Ví dụ 1 bằng tay

`s = "rabbbit"` có 3 chữ `b` ở chỉ số `2, 3, 4`. `t = "rabbit"` chỉ cần **2** chữ `b`.

- Chữ `r`, `a`, `i`, `t` trong `s` đều xuất hiện đúng một lần → bắt buộc chọn.
- Ta chỉ cần chọn **2 trong 3** chữ `b`: `{2,3}`, `{2,4}`, `{3,4}` → đúng **3** cách.

Ba cách đều tạo ra **cùng một chuỗi** `"rabbit"`, nhưng chúng là 3 cách **khác nhau** vì tập chỉ số được chọn khác nhau. Đây chính là ý nghĩa của "distinct" trong đề — ta sẽ làm rõ ở mục 2.

Ví dụ 2 (`"babgbag"`, `"bag"`) khó đếm nhẩm hơn nhiều — đó là lý do ta cần thuật toán.

## 2. Kiến thức nền: subsequence khác substring thế nào?

- **Substring (chuỗi con)**: một đoạn **liên tiếp** của chuỗi. Ví dụ với `"abc"`: `"ab"`, `"bc"`, `"abc"`... Lấy bằng slicing `s[i:j]`.
- **Subsequence (dãy con)**: xóa đi **một số** ký tự (có thể 0 ký tự), **giữ nguyên thứ tự** các ký tự còn lại. Không cần liên tiếp. Ví dụ với `"abc"`: `"ac"` là subsequence (bỏ `b`) nhưng **không** phải substring.
- `"ca"` **không** phải subsequence của `"abc"`, vì thứ tự bị đảo.

Một cách nghĩ dễ nhớ: subsequence = **chọn một tập chỉ số tăng dần** `i1 < i2 < ... < ik` rồi đọc `s[i1] s[i2] ... s[ik]`.

Với chuỗi dài `n`, mỗi ký tự có 2 lựa chọn (giữ / bỏ) nên có `2^n` cách chọn (kể cả chọn rỗng). Còn substring chỉ có khoảng `n(n+1)/2` đoạn.

In [2]:
s = "abc"

substrings = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print("Substring  :", substrings)

# subsequence: mỗi ký tự giữ (bit 1) hoặc bỏ (bit 0) -> 2^3 = 8 cách
subsequences = []
for mask in range(2 ** len(s)):
    subsequences.append("".join(s[i] for i in range(len(s)) if mask >> i & 1))
print("Subsequence:", subsequences)

print("'ac' là substring?   ", "ac" in substrings)
print("'ac' là subsequence? ", "ac" in subsequences)
print("'ca' là subsequence? ", "ca" in subsequences)

Substring  : ['a', 'ab', 'abc', 'b', 'bc', 'c']
Subsequence: ['', 'a', 'b', 'ab', 'c', 'ac', 'bc', 'abc']
'ac' là substring?    False
'ac' là subsequence?  True
'ca' là subsequence?  False


### Kiểm tra "`t` có là subsequence của `s` không?" bằng hai con trỏ

Trước khi **đếm**, hãy biết cách **kiểm tra** (đây chính là LeetCode 392 — Is Subsequence).

Ý tưởng **hai con trỏ (two pointers)**: con trỏ `j` chỉ vào ký tự cần tìm tiếp theo của `t`. Duyệt `s` từ trái sang phải; mỗi khi `s[i] == t[j]` thì tăng `j`. Cuối cùng nếu `j == len(t)` thì đã tìm đủ.

Ví dụ `s = "babgbag"`, `t = "bag"`: gặp `b` (i=0) → tìm `a`; gặp `a` (i=1) → tìm `g`; ... gặp `g` (i=3) → xong. Kết quả `True`.

Nhưng hai con trỏ chỉ tìm **một** cách chọn (cách "sớm nhất"). Bài 115 hỏi **có bao nhiêu** cách — khó hơn hẳn.

In [3]:
def is_subsequence(t: str, s: str) -> bool:
    j = 0                          # vị trí cần tìm tiếp theo trong t
    for ch in s:
        if j < len(t) and ch == t[j]:
            j += 1
    return j == len(t)


print(is_subsequence("bag", "babgbag"))   # True
print(is_subsequence("ac", "abc"))        # True
print(is_subsequence("ca", "abc"))        # False

True
True
False


### Chữ "distinct" nghĩa là gì? Cạm bẫy số 1 của bài

Người mới thường đọc "distinct subsequences" là "các **chuỗi** subsequence khác nhau" — và định dùng `set` để loại trùng. **Sai!**

Đề đếm **các cách chọn tập chỉ số khác nhau** sao cho chuỗi đọc ra bằng `t`. Vì mọi cách đều đọc ra cùng chuỗi `t`, nếu đếm theo chuỗi thì đáp án luôn chỉ là 0 hoặc 1 — bài toán sẽ chẳng có gì để hỏi.

Ví dụ `s = "aaa"`, `t = "aa"`:

| Tập chỉ số | Chuỗi đọc ra |
|---|---|
| `(0, 1)` | `"aa"` |
| `(0, 2)` | `"aa"` |
| `(1, 2)` | `"aa"` |

→ Đáp án là **3** (ba cách chọn), không phải 1.

In [4]:
s, t = "aaa", "aa"
index_choices = [(0, 1), (0, 2), (1, 2)]
strings = ["".join(s[i] for i in c) for c in index_choices]

print("Các cách chọn chỉ số:", index_choices)
print("Chuỗi đọc ra        :", strings)
print("Đếm theo cách chọn  :", sum(x == t for x in strings), "<- đúng ý đề")
print("Đếm theo set chuỗi  :", len({x for x in strings if x == t}), "<- hiểu sai đề")

Các cách chọn chỉ số: [(0, 1), (0, 2), (1, 2)]
Chuỗi đọc ra        : ['aa', 'aa', 'aa']
Đếm theo cách chọn  : 3 <- đúng ý đề
Đếm theo set chuỗi  : 1 <- hiểu sai đề


### Công cụ: `itertools.combinations`

Muốn liệt kê mọi tập chỉ số **tăng dần** có đúng `k` phần tử, Python có sẵn `itertools.combinations(iterable, k)`:

- Nó sinh ra mọi **tổ hợp (combination)** chọn `k` phần tử, mỗi tổ hợp là một `tuple`.
- Các phần tử trong mỗi tuple **giữ thứ tự** như trong `iterable` → với `range(n)` thì tuple luôn tăng dần — đúng là một cách chọn subsequence!
- Số tổ hợp là `C(n, k) = n! / (k!·(n-k)!)`, tính bằng `math.comb(n, k)`.

Ví dụ chọn 2 trong 4 chỉ số `0..3`: có `C(4,2) = 6` tổ hợp.

In [5]:
from itertools import combinations
import math

print(list(combinations(range(4), 2)))
print("math.comb(4, 2) =", math.comb(4, 2))

# Áp dụng: mọi subsequence độ dài 3 của "babg"
s = "babg"
for idx in combinations(range(len(s)), 3):
    print(idx, "->", "".join(s[i] for i in idx))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
math.comb(4, 2) = 6
(0, 1, 2) -> bab
(0, 1, 3) -> bag
(0, 2, 3) -> bbg
(1, 2, 3) -> abg


## 3. Cách 1 — Brute force: thử mọi bộ chỉ số

Cách người mới tự nghĩ ra đầu tiên: subsequence bằng `t` thì **phải có đúng `len(t)` ký tự**. Vậy:

1. Sinh mọi bộ chỉ số tăng dần có `m = len(t)` phần tử bằng `combinations(range(n), m)`.
2. Với mỗi bộ, ghép chuỗi `s[i1] s[i2] ...` và so với `t`.
3. Đếm số bộ khớp.

Đơn giản, đúng định nghĩa, và rất hữu ích làm **"trọng tài"** để kiểm tra các cách thông minh hơn trên input nhỏ.

In [6]:
from itertools import combinations


def num_distinct_bruteforce(s: str, t: str) -> int:
    m = len(t)
    count = 0
    for idx in combinations(range(len(s)), m):      # mọi bộ chỉ số tăng dần, độ dài m
        if all(s[i] == t[k] for k, i in enumerate(idx)):
            count += 1
    return count


print(num_distinct_bruteforce("rabbbit", "rabbit"))   # 3
print(num_distinct_bruteforce("babgbag", "bag"))      # 5
print(num_distinct_bruteforce("aaa", "aa"))           # 3
print(num_distinct_bruteforce("abc", "abcd"))         # 0 (t dài hơn s)

# Số bộ chỉ số brute force phải thử là C(n, m): in số chữ số của nó
for n, m in [(7, 6), (20, 10), (40, 20), (1000, 500)]:
    c = math.comb(n, m)
    print(f"n={n:4}, m={m:3}: C(n, m) có {len(str(c)):3} chữ số")

3
5
3
0
n=   7, m=  6: C(n, m) có   1 chữ số
n=  20, m= 10: C(n, m) có   6 chữ số
n=  40, m= 20: C(n, m) có  12 chữ số
n=1000, m=500: C(n, m) có 300 chữ số


### Brute force vướng ở đâu?

Số bộ chỉ số phải thử là `C(n, m)`. Bảng in ở trên cho thấy: với `n = 1000`, `m = 500` (vẫn hợp lệ theo constraints), `C(n, m)` có tới **300 chữ số**. Ta không thể liệt kê từng cách khi **chính số cách** đã lớn như vậy — kể cả máy tính nhanh nhất thế giới chạy từ thuở vũ trụ hình thành cũng không xong.

**Độ phức tạp Cách 1:** Time `O(C(n, m) · m)` (mỗi bộ so sánh tối đa `m` ký tự), Space `O(m)` cho một bộ chỉ số.

**Nút thắt:** brute force đếm **từng cách một**. Muốn nhanh, ta phải đếm **theo nhóm**: nhiều cách chọn có chung phần đuôi, nên chỉ cần biết "phần còn lại có bao nhiêu cách" rồi cộng dồn. Ý tưởng "chia bài toán thành bài toán con" này dẫn ta tới **đệ quy**.

## 4. Kiến thức nền: đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** để giải một bài toán **nhỏ hơn** cùng loại. Mỗi hàm đệ quy có 2 phần:

1. **Base case (trường hợp cơ sở):** bài toán đủ nhỏ để trả lời ngay. Thiếu nó, hàm gọi mãi không dừng.
2. **Recursive case (trường hợp đệ quy):** làm một bước nhỏ, giao phần còn lại cho chính hàm đó.

Ví dụ mini: đếm độ dài chuỗi mà không dùng `len`.

- Base case: chuỗi rỗng có độ dài 0.
- Recursive case: độ dài `"abc"` = 1 + độ dài `"bc"`.

Thay vì cắt chuỗi `s[1:]` (tạo chuỗi mới, tốn bộ nhớ), ta quen dần với cách truyền **chỉ số `i`**: "xét phần `s[i:]`". Đây là cách ta sẽ dùng trong bài.

In [7]:
def length_rec(s: str, i: int = 0) -> int:
    # độ dài của phần s[i:]
    if i == len(s):                 # base case: phần còn lại rỗng
        return 0
    return 1 + length_rec(s, i + 1) # recursive case: 1 ký tự + phần sau


print(length_rec("abc"))     # 3
print(length_rec(""))        # 0

3
0


### Đệ quy "chọn / không chọn": cây quyết định

Đây là khuôn mẫu quan trọng nhất của bài. Để sinh mọi subsequence của `"abc"`, tại mỗi vị trí `i` ta có đúng **2 lựa chọn**:

- **Không chọn** `s[i]`: đi tiếp sang `i + 1`, chuỗi đang xây giữ nguyên.
- **Chọn** `s[i]`: đi tiếp sang `i + 1`, chuỗi đang xây thêm `s[i]`.

Khi `i == len(s)` (base case) ta có một subsequence hoàn chỉnh. Các lựa chọn tạo thành một **cây nhị phân** có `2^3 = 8` lá. Hàm dưới in cây đó, thụt lề theo độ sâu.

In [8]:
def all_subsequences(s: str, i: int = 0, built: str = "", depth: int = 0) -> list:
    pad = "    " * depth
    if i == len(s):
        print(f"{pad}lá: {built!r}")
        return [built]
    print(f"{pad}i={i} ('{s[i]}'), đang có {built!r}")
    skip = all_subsequences(s, i + 1, built, depth + 1)          # không chọn s[i]
    take = all_subsequences(s, i + 1, built + s[i], depth + 1)   # chọn s[i]
    return skip + take


result = all_subsequences("ab")
print("Tất cả subsequence của 'ab':", result)

i=0 ('a'), đang có ''
    i=1 ('b'), đang có ''
        lá: ''
        lá: 'b'
    i=1 ('b'), đang có 'a'
        lá: 'a'
        lá: 'ab'
Tất cả subsequence của 'ab': ['', 'b', 'a', 'ab']


## 5. Cách 2 — Đệ quy chọn / không chọn `s[i]`

Thay vì sinh hết subsequence rồi so sánh, ta **đi song song trên `s` và `t`** và chỉ "chọn" một ký tự khi nó **khớp** ký tự cần tìm của `t`.

**Trạng thái (state):** `f(i, j)` = số cách chọn subsequence trong phần `s[i:]` sao cho bằng phần `t[j:]`.

Đáp án của bài là `f(0, 0)`.

**Tại mỗi bước, xét ký tự `s[i]`:**

- **Luôn có lựa chọn bỏ qua `s[i]`:** số cách là `f(i + 1, j)` (vẫn đang cần `t[j]`).
- **Nếu `s[i] == t[j]`, có thêm lựa chọn dùng `s[i]` để khớp `t[j]`:** số cách là `f(i + 1, j + 1)`.

Vì hai lựa chọn tạo ra các tập chỉ số **khác nhau** (một bên có chỉ số `i`, một bên không) nên tổng số cách là **phép cộng**:

```text
f(i, j) = f(i+1, j) + (f(i+1, j+1)  nếu s[i] == t[j], ngược lại 0)
```

**Base case — thứ tự kiểm tra rất quan trọng:**

1. `j == len(t)`: đã khớp hết `t` → có đúng **1** cách (từ đây không chọn gì thêm). Trả `1`.
2. `i == len(s)` (mà `j` chưa hết): `s` cạn mà `t` còn ký tự → **0** cách.

Phải kiểm tra `j == len(t)` **trước**: khi cả hai cùng hết (`i == len(s)` và `j == len(t)`) thì đó là một cách hợp lệ, phải trả `1` chứ không phải `0`.

**Ví dụ số:** `s = "aa"`, `t = "a"`. `f(0,0)`: `s[0] == t[0]` nên `f(0,0) = f(1,0) + f(1,1)`. `f(1,1) = 1` (đã khớp hết `t`). `f(1,0) = f(2,0) + f(2,1) = 0 + 1 = 1`. Tổng `f(0,0) = 2` — đúng: chọn chữ `a` thứ nhất hoặc thứ hai.

In [9]:
def num_distinct_recursive(s: str, t: str) -> int:
    n, m = len(s), len(t)

    def f(i: int, j: int) -> int:
        if j == m:              # đã khớp hết t: 1 cách
            return 1
        if i == n:              # s cạn mà t chưa hết: 0 cách
            return 0
        ways = f(i + 1, j)                  # bỏ qua s[i]
        if s[i] == t[j]:
            ways += f(i + 1, j + 1)         # dùng s[i] khớp t[j]
        return ways

    return f(0, 0)


print(num_distinct_recursive("aa", "a"))            # 2
print(num_distinct_recursive("rabbbit", "rabbit"))  # 3
print(num_distinct_recursive("babgbag", "bag"))     # 5

2
3
5


### Trace Cách 2: cây lời gọi đệ quy

Hàm `trace_recursive` dưới đây giống hệt Cách 2 nhưng in mỗi lời gọi `f(i, j)`, thụt lề theo độ sâu, kèm lý do. Input: `s = "bbag"`, `t = "bag"` (đáp án 2: chữ `b` ở chỉ số 0 hoặc chỉ số 1, rồi `a`, `g`).

Cách đọc: mỗi dòng `f(i,j)` cho biết đang xét `s[i]` và cần tìm `t[j]`; dòng `= số` là kết quả trả về.

In [10]:
def trace_recursive(s: str, t: str, i: int = 0, j: int = 0, depth: int = 0) -> int:
    pad = "  " * depth
    if j == len(t):
        print(f"{pad}f({i},{j}): đã khớp hết t -> 1")
        return 1
    if i == len(s):
        print(f"{pad}f({i},{j}): s cạn, còn thiếu {t[j:]!r} -> 0")
        return 0
    match = s[i] == t[j]
    print(f"{pad}f({i},{j}): s[{i}]='{s[i]}' vs t[{j}]='{t[j]}' {'khớp' if match else 'không khớp'}")
    ways = trace_recursive(s, t, i + 1, j, depth + 1)
    if match:
        ways += trace_recursive(s, t, i + 1, j + 1, depth + 1)
    print(f"{pad}= {ways}")
    return ways


print("Đáp án:", trace_recursive("bbag", "bag"))

f(0,0): s[0]='b' vs t[0]='b' khớp
  f(1,0): s[1]='b' vs t[0]='b' khớp
    f(2,0): s[2]='a' vs t[0]='b' không khớp
      f(3,0): s[3]='g' vs t[0]='b' không khớp
        f(4,0): s cạn, còn thiếu 'bag' -> 0
      = 0
    = 0
    f(2,1): s[2]='a' vs t[1]='a' khớp
      f(3,1): s[3]='g' vs t[1]='a' không khớp
        f(4,1): s cạn, còn thiếu 'ag' -> 0
      = 0
      f(3,2): s[3]='g' vs t[2]='g' khớp
        f(4,2): s cạn, còn thiếu 'g' -> 0
        f(4,3): đã khớp hết t -> 1
      = 1
    = 1
  = 1
  f(1,1): s[1]='b' vs t[1]='a' không khớp
    f(2,1): s[2]='a' vs t[1]='a' khớp
      f(3,1): s[3]='g' vs t[1]='a' không khớp
        f(4,1): s cạn, còn thiếu 'ag' -> 0
      = 0
      f(3,2): s[3]='g' vs t[2]='g' khớp
        f(4,2): s cạn, còn thiếu 'g' -> 0
        f(4,3): đã khớp hết t -> 1
      = 1
    = 1
  = 1
= 2
Đáp án: 2


### Đọc trace `("bbag", "bag")`

| Lời gọi | Đang xét | Lựa chọn | Kết quả |
|---|---|---|---|
| `f(0,0)` | `s[0]='b'` khớp `t[0]='b'` | bỏ → `f(1,0)`; dùng → `f(1,1)` | `1 + 1 = 2` |
| `f(1,0)` | `s[1]='b'` khớp `t[0]='b'` | bỏ → `f(2,0)` = 0; dùng → `f(2,1)` = 1 | `1` |
| `f(1,1)` | `s[1]='b'` không khớp `t[1]='a'` | chỉ bỏ → `f(2,1)` | `1` |
| `f(2,1)` | `s[2]='a'` khớp `t[1]='a'` | bỏ → `f(3,1)` = 0; dùng → `f(3,2)` = 1 | `1` |

Nhìn kỹ: **`f(2,1)` được tính 2 lần** (một lần trong nhánh `f(1,0)`, một lần trong nhánh `f(1,1)`). Với input nhỏ thì không sao, nhưng với input lớn, các bài toán con trùng lặp nhân lên theo cấp số nhân. Cell dưới đo điều đó.

In [11]:
from collections import Counter


def count_calls_recursive(s: str, t: str):
    calls = Counter()            # đếm số lần mỗi trạng thái (i, j) được gọi
    n, m = len(s), len(t)

    def f(i, j):
        calls[(i, j)] += 1
        if j == m:
            return 1
        if i == n:
            return 0
        ways = f(i + 1, j)
        if s[i] == t[j]:
            ways += f(i + 1, j + 1)
        return ways

    return f(0, 0), calls


print(" n | đáp án | tổng số lời gọi | số trạng thái khác nhau")
for n in [8, 12, 16, 20]:
    ans, calls = count_calls_recursive("a" * n, "a" * (n // 2))
    print(f"{n:2} | {ans:6} | {sum(calls.values()):15} | {len(calls):5}")

 n | đáp án | tổng số lời gọi | số trạng thái khác nhau
 8 |     70 |             325 |    35
12 |    924 |            5019 |    70
16 |  12870 |           78405 |   117


20 | 184756 |         1233331 |   176


Với `s = "a"*20`, `t = "a"*10`: hơn **một triệu** lời gọi nhưng chỉ có **176** trạng thái `(i, j)` khác nhau! Nghĩa là cùng một câu hỏi "`s[i:]` tạo `t[j:]` bao nhiêu cách?" bị hỏi đi hỏi lại hàng nghìn lần.

**Độ phức tạp Cách 2:** Time **mũ** (exponential) theo `n` trong trường hợp xấu — cỡ `C(n, m)` lời gọi tới nhánh thành công, thậm chí nhiều hơn. Space `O(n)` cho call stack (độ sâu tối đa `n + 1`).

**Nút thắt:** tính lại các bài toán con trùng lặp (**overlapping subproblems**). **Mấu chốt cải tiến:** mỗi `(i, j)` chỉ có một đáp án duy nhất → tính một lần, **ghi nhớ** lại, lần sau tra cứu. Đó là **memoization**.

## 6. Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = dùng một "cuốn sổ" (thường là `dict`) lưu kết quả của mỗi bài toán con. Trước khi tính, tra sổ: có rồi thì trả luôn; chưa có thì tính, ghi vào sổ, rồi trả.

Ví dụ kinh điển: dãy Fibonacci `fib(n) = fib(n-1) + fib(n-2)`, `fib(0)=0`, `fib(1)=1`. Đệ quy thuần tính `fib(n-2)` hai lần, `fib(n-3)` ba lần... Ta đếm số lời gọi của hai phiên bản.

In [12]:
calls_plain = 0
def fib_plain(n):
    global calls_plain
    calls_plain += 1
    if n < 2:
        return n
    return fib_plain(n - 1) + fib_plain(n - 2)


calls_memo = 0
def fib_memo(n, memo):
    global calls_memo
    calls_memo += 1
    if n in memo:                  # tra sổ trước
        return memo[n]
    result = n if n < 2 else fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    memo[n] = result               # ghi vào sổ
    return result


print("fib(25) =", fib_plain(25), "| lời gọi đệ quy thuần:", calls_plain)
print("fib(25) =", fib_memo(25, {}), "| lời gọi có memo     :", calls_memo)

fib(25) = 75025 | lời gọi đệ quy thuần: 242785
fib(25) = 75025 | lời gọi có memo     : 49


### `functools.lru_cache` / `functools.cache`: memo tự động

Viết `memo` bằng tay hơi dài. Python có sẵn **decorator** `@lru_cache(maxsize=None)` (hoặc `@cache`, Python 3.9+) trong module `functools`:

- **Decorator** là dòng `@...` đặt ngay trên `def`, nó "bọc" hàm lại để thêm hành vi — ở đây là tự động nhớ kết quả theo **bộ tham số**.
- Điều kiện: các tham số phải **hashable** (băm được: `int`, `str`, `tuple`... được; `list`, `dict` thì không).
- `f.cache_info()` cho biết số lần tra trúng (`hits`), số lần phải tính (`misses`).

In [13]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(80))
print(fib.cache_info())   # misses = 81: mỗi n từ 0..80 chỉ tính đúng một lần

23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


## 7. Cách 3 — Đệ quy + memo (top-down DP)

Áp dụng vào bài: giữ nguyên hàm `f(i, j)` của Cách 2, chỉ thêm `@lru_cache(maxsize=None)`. Tham số `(i, j)` là hai số nguyên → hashable.

Đây gọi là **top-down DP (quy hoạch động từ trên xuống)**: bắt đầu từ câu hỏi lớn `f(0, 0)`, đệ quy xuống các câu hỏi nhỏ, và nhớ lại mọi đáp án đã có.

Thêm một cải tiến nhỏ nhưng hữu ích — **cắt tỉa (pruning)**: nếu số ký tự còn lại của `s` (`n - i`) **ít hơn** số ký tự còn cần của `t` (`m - j`) thì chắc chắn 0 cách, trả luôn.

In [14]:
from functools import lru_cache


def num_distinct_memo(s: str, t: str) -> int:
    n, m = len(s), len(t)

    @lru_cache(maxsize=None)
    def f(i: int, j: int) -> int:
        if j == m:
            return 1
        if n - i < m - j:           # pruning: không đủ ký tự (bao gồm cả trường hợp i == n)
            return 0
        ways = f(i + 1, j)
        if s[i] == t[j]:
            ways += f(i + 1, j + 1)
        return ways

    return f(0, 0)


print(num_distinct_memo("rabbbit", "rabbit"))   # 3
print(num_distinct_memo("babgbag", "bag"))      # 5
print(num_distinct_memo("a" * 20, "a" * 10))    # 184756 = C(20, 10), tính tức thì

3
5
184756


Để ý dòng pruning `n - i < m - j` đã **bao luôn** base case `i == n` (khi đó `n - i = 0 < m - j` vì `j < m`). Nên không cần viết riêng.

So sánh số lần thực sự phải tính (số `misses` của cache) với số lời gọi của Cách 2:

In [15]:
def memo_misses(s: str, t: str) -> int:
    n, m = len(s), len(t)

    @lru_cache(maxsize=None)
    def f(i, j):
        if j == m:
            return 1
        if i == n:
            return 0
        return f(i + 1, j) + (f(i + 1, j + 1) if s[i] == t[j] else 0)

    f(0, 0)
    return f.cache_info().misses


print(" n | lời gọi Cách 2 | trạng thái tính ở Cách 3")
for n in [8, 12, 16, 20]:
    s, t = "a" * n, "a" * (n // 2)
    _, calls = count_calls_recursive(s, t)
    print(f"{n:2} | {sum(calls.values()):14} | {memo_misses(s, t):5}")

 n | lời gọi Cách 2 | trạng thái tính ở Cách 3
 8 |            325 |    35
12 |           5019 |    70
16 |          78405 |   117


20 |        1233331 |   176


### Cạm bẫy: giới hạn độ sâu đệ quy của Python

Mỗi lời gọi `f(i, j)` gọi tiếp `f(i + 1, ...)` → chuỗi lời gọi lồng nhau sâu tới `n + 1` tầng. Với `n = 1000` (đúng giới hạn của đề) là **1001 tầng**.

Python mặc định chỉ cho khoảng **1000** tầng (`sys.getrecursionlimit()`), vượt quá sẽ ném `RecursionError`. Nghĩa là Cách 3 có thể **chết ngay trên input hợp lệ**! Thử với `s` = 1000 chữ `a`, `t = "b"` (đáp án 0, nhưng phải đi hết chiều sâu).

Cách chữa: tăng giới hạn bằng `sys.setrecursionlimit(...)` (nhiều người nộp LeetCode vẫn làm vậy), hoặc tốt hơn: **bỏ hẳn đệ quy** bằng bảng DP bottom-up ở Cách 4.

In [16]:
import sys

print("Giới hạn mặc định:", sys.getrecursionlimit())
big_s, big_t = "a" * 1000, "b"
try:
    print(num_distinct_memo(big_s, big_t))
except RecursionError as e:
    print("RecursionError:", e)

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(10_000)
print("Sau khi tăng giới hạn:", num_distinct_memo(big_s, big_t))
sys.setrecursionlimit(old_limit)       # trả lại như cũ cho các cell sau

Giới hạn mặc định: 1000
RecursionError: maximum recursion depth exceeded
Sau khi tăng giới hạn: 0


**Độ phức tạp Cách 3** (`n = len(s)`, `m = len(t)`):

- Có tối đa `(n + 1)·(m + 1)` trạng thái, mỗi trạng thái tính `O(1)` (một phép so sánh, hai lần tra) → **Time `O(n·m)`**. Với `n = m = 1000` là khoảng 10⁶ trạng thái.
- **Space `O(n·m)`** cho cache, cộng `O(n)` cho call stack.

**Còn vướng gì?** (1) độ sâu đệ quy như vừa thấy; (2) gọi hàm và tra cache trong Python khá chậm, cache 10⁶ phần tử tốn nhiều bộ nhớ. **Mấu chốt cải tiến:** ta biết rõ `f(i, j)` chỉ phụ thuộc `f(i+1, ·)` → có thể **tự điền bảng theo đúng thứ tự** bằng vòng lặp, không cần đệ quy. Đó là **bottom-up DP**.

## 8. Kiến thức nền: bảng 2 chiều (list lồng list) trong Python

Bảng DP là một **list các list**: `dp[i][j]` đọc là "hàng `i`, cột `j`".

Cách tạo **đúng**: list comprehension `[[0] * (m + 1) for _ in range(n + 1)]` — mỗi hàng là một list **riêng**.

Cách tạo **sai** rất phổ biến: `[[0] * (m + 1)] * (n + 1)` — phép `*` trên list ngoài **chép tham chiếu**, tất cả các hàng là **cùng một list**. Sửa một ô là sửa cả cột!

In [17]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2

good[0][0] = 1
bad[0][0] = 1
print("good:", good)     # [[1, 0, 0], [0, 0, 0]]
print("bad :", bad)      # [[1, 0, 0], [1, 0, 0]]  <- hàng 1 cũng bị đổi!
print("Hai hàng của bad là cùng một object?", bad[0] is bad[1])

good: [[1, 0, 0], [0, 0, 0]]
bad : [[1, 0, 0], [1, 0, 0]]
Hai hàng của bad là cùng một object? True


## 9. Cách 4 — DP bảng `dp[i][j]` bottom-up

Ở bảng, ta quen dùng **tiền tố (prefix)** thay cho hậu tố:

> `dp[i][j]` = số cách chọn subsequence trong **`i` ký tự đầu** của `s` (tức `s[:i]`) sao cho bằng **`j` ký tự đầu** của `t` (tức `t[:j]`).

Đáp án là `dp[n][m]`. Bảng có kích thước `(n + 1) × (m + 1)` vì `i`, `j` chạy từ `0` (tiền tố rỗng) tới hết.

**Chú ý lệch chỉ số:** ô `dp[i][j]` xét ký tự **cuối** của hai tiền tố, đó là `s[i-1]` và `t[j-1]` (không phải `s[i]`, `t[j]`).

**Công thức chuyển (transition)** — cùng tư duy chọn / không chọn, nhưng xét ký tự **cuối** `s[i-1]`:

```text
dp[i][j] = dp[i-1][j]                          # không dùng s[i-1]
         + (dp[i-1][j-1]  nếu s[i-1] == t[j-1]) # dùng s[i-1] khớp t[j-1]
```

**Base case:**

- `dp[i][0] = 1` với mọi `i`: tạo `t` rỗng từ bất kỳ tiền tố nào có đúng 1 cách — không chọn gì cả. (Đây là "hạt giống" của mọi phép đếm; quên nó thì cả bảng bằng 0.)
- `dp[0][j] = 0` với `j ≥ 1`: từ `s` rỗng không thể tạo `t` khác rỗng.

**Ví dụ số** (`s = "rabbbit"`, `t = "rabbit"`): ô `dp[4][4]` hỏi "`"rabb"` tạo `"rabb"` mấy cách?". `s[3] = 'b'` khớp `t[3] = 'b'` → `dp[4][4] = dp[3][4] + dp[3][3] = 0 + 1 = 1`. Ô `dp[5][4]` ("`rabbb`" tạo "`rabb`"): `s[4]='b'` khớp `t[3]='b'` → `dp[4][4] + dp[4][3] = 1 + 2 = 3`. Đúng: chọn 2 trong 3 chữ `b`.

**Thứ tự điền:** ô `(i, j)` chỉ cần hàng `i - 1` → điền **từ trên xuống** (i tăng dần) là luôn có đủ dữ liệu.

In [18]:
def num_distinct_dp(s: str, t: str) -> int:
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = 1                       # t rỗng: 1 cách (không chọn gì)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j]        # không dùng s[i-1]
            if s[i - 1] == t[j - 1]:
                dp[i][j] += dp[i - 1][j - 1]   # dùng s[i-1] khớp t[j-1]
    return dp[n][m]


print(num_distinct_dp("rabbbit", "rabbit"))   # 3
print(num_distinct_dp("babgbag", "bag"))      # 5
print(num_distinct_dp("a" * 20, "a" * 10))    # 184756

3
5
184756


### Nhìn bảng DP

Hàm `build_dp_table` trả về cả bảng, `print_dp_table` in bảng với tiêu đề hàng là ký tự `s[i-1]`, tiêu đề cột là ký tự `t[j-1]` (`-` là tiền tố rỗng).

In [19]:
def build_dp_table(s: str, t: str) -> list:
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = 1
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j] + (dp[i - 1][j - 1] if s[i - 1] == t[j - 1] else 0)
    return dp


def print_dp_table(s: str, t: str) -> None:
    dp = build_dp_table(s, t)
    print("        " + "  ".join(f"{c:>2}" for c in "-" + t))
    for i, row in enumerate(dp):
        label = "-" if i == 0 else s[i - 1]
        print(f"i={i} {label:>2} " + "  ".join(f"{v:>2}" for v in row))


print_dp_table("rabbbit", "rabbit")

         -   r   a   b   b   i   t
i=0  -  1   0   0   0   0   0   0
i=1  r  1   1   0   0   0   0   0
i=2  a  1   1   1   0   0   0   0
i=3  b  1   1   1   1   0   0   0
i=4  b  1   1   1   2   1   0   0
i=5  b  1   1   1   3   3   0   0
i=6  i  1   1   1   3   3   3   0
i=7  t  1   1   1   3   3   3   3


### Đọc bảng `s = "rabbbit"`, `t = "rabbit"`

| i \ cột | `-` | `r` | `a` | `b` | `b` | `i` | `t` |
|---|---|---|---|---|---|---|---|
| 0 `-` | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| 1 `r` | 1 | 1 | 0 | 0 | 0 | 0 | 0 |
| 2 `a` | 1 | 1 | 1 | 0 | 0 | 0 | 0 |
| 3 `b` | 1 | 1 | 1 | **1** | 0 | 0 | 0 |
| 4 `b` | 1 | 1 | 1 | **2** | **1** | 0 | 0 |
| 5 `b` | 1 | 1 | 1 | **3** | **3** | 0 | 0 |
| 6 `i` | 1 | 1 | 1 | 3 | 3 | 3 | 0 |
| 7 `t` | 1 | 1 | 1 | 3 | 3 | 3 | **3** |

Những điểm đáng chú ý:

- **Cột 0 toàn số 1** (hạt giống). Hàng 0 (trừ ô đầu) toàn 0.
- **Cột `b` thứ nhất (j=3)** tăng 1 → 2 → 3 ở các hàng 3, 4, 5: "`rab`" có 1, 2, 3 cách lần lượt khi `s` có 1, 2, 3 chữ `b` — mỗi chữ `b` mới đều có thể làm chữ `b` của `t`.
- **Cột `b` thứ hai (j=4)**: 0 → 1 → 3 — đây chính là `C(1,2)=0`, `C(2,2)=1`, `C(3,2)=3`: chọn 2 chữ `b` trong số các chữ `b` đã gặp.
- Từ hàng 6 trở đi giá trị `3` chỉ được **"chuyển chéo"** xuống: `i` khớp `i`, `t` khớp `t`, mỗi lần đúng 1 lựa chọn.
- Đáp án `dp[7][6] = 3` ở góc dưới phải.

Một ô có hai nguồn: **ô ngay trên** (bỏ ký tự `s[i-1]`) và, nếu khớp, **ô chéo trên-trái** (dùng nó). Ô không khớp chỉ đơn giản là chép ô ngay trên xuống.

### Trace Cách 4: chỉ in các ô có ký tự khớp

Các ô không khớp chỉ chép ô trên, nên hàm `trace_dp` chỉ in những ô `s[i-1] == t[j-1]` — nơi phép cộng thật sự xảy ra — cùng hai thành phần của phép cộng.

In [20]:
def trace_dp(s: str, t: str) -> int:
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = 1
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j]
            if s[i - 1] == t[j - 1]:
                up, diag = dp[i - 1][j], dp[i - 1][j - 1]
                dp[i][j] += diag
                print(f"dp[{i}][{j}]: s[{i-1}]='{s[i-1]}' khớp t[{j-1}]='{t[j-1]}' "
                      f"-> trên {up} + chéo {diag} = {dp[i][j]}")
    print("Đáp án dp[n][m] =", dp[n][m])
    return dp[n][m]


trace_dp("rabbbit", "rabbit")

dp[1][1]: s[0]='r' khớp t[0]='r' -> trên 0 + chéo 1 = 1
dp[2][2]: s[1]='a' khớp t[1]='a' -> trên 0 + chéo 1 = 1
dp[3][3]: s[2]='b' khớp t[2]='b' -> trên 0 + chéo 1 = 1
dp[3][4]: s[2]='b' khớp t[3]='b' -> trên 0 + chéo 0 = 0
dp[4][3]: s[3]='b' khớp t[2]='b' -> trên 1 + chéo 1 = 2
dp[4][4]: s[3]='b' khớp t[3]='b' -> trên 0 + chéo 1 = 1
dp[5][3]: s[4]='b' khớp t[2]='b' -> trên 2 + chéo 1 = 3
dp[5][4]: s[4]='b' khớp t[3]='b' -> trên 1 + chéo 2 = 3
dp[6][5]: s[5]='i' khớp t[4]='i' -> trên 0 + chéo 3 = 3
dp[7][6]: s[6]='t' khớp t[5]='t' -> trên 0 + chéo 3 = 3
Đáp án dp[n][m] = 3


3

### Liên hệ Cách 3 và Cách 4

Cách 3 dùng **hậu tố** (`s[i:]`, `t[j:]`) và đi từ `f(0,0)` xuống; Cách 4 dùng **tiền tố** (`s[:i]`, `t[:j]`) và điền từ hàng 0 lên. Hai cách nhìn tương đương: đếm subsequence từ trái sang hay phải sang đều cho cùng số cách. Bạn có thể chọn kiểu nào cũng được, miễn **nhất quán** giữa định nghĩa, base case và thứ tự điền.

**Độ phức tạp Cách 4:** Time `O(n·m)` (hai vòng lặp lồng nhau, mỗi ô `O(1)`), Space `O(n·m)` cho bảng. Không còn đệ quy → không lo `RecursionError`.

### Chú ý: giá trị trung gian có thể rất lớn

Đề chỉ hứa **đáp án** vừa 32-bit. Các ô giữa bảng có thể lớn hơn nhiều! Ví dụ `s = "a"*100 + "b"`, `t = "a"*50 + "c"`: đáp án là 0 (không có `c`), nhưng ô `dp[100][50] = C(100, 50)` khổng lồ.

Python có số nguyên không giới hạn nên không sao. Trong C++/Java, ô trung gian có thể vượt cả 64-bit, nên người ta phải dùng kiểu lớn hơn (`unsigned long long`, `double`...) và chấp nhận các ô "không dùng tới" bị tràn mà không ảnh hưởng đáp án — một chủ đề tranh luận quen thuộc ở phần Discuss của bài. Đây là lý do bài này nổi tiếng hay bị **tràn số (overflow)** ở các ngôn ngữ khác.

In [21]:
s, t = "a" * 100 + "b", "a" * 50 + "c"
dp = build_dp_table(s, t)
biggest = max(max(row) for row in dp)
print("Đáp án        :", dp[-1][-1])
print("Ô lớn nhất    :", biggest)
print("Vượt 32-bit?  :", biggest > 2**31 - 1)

Đáp án        : 0
Ô lớn nhất    : 100891344545564193334812497256
Vượt 32-bit?  : True


## 10. Cách 5 — DP 1 chiều, duyệt `j` ngược

### Quan sát: mỗi hàng chỉ cần hàng ngay trên

Công thức `dp[i][j] = dp[i-1][j] + dp[i-1][j-1]·[khớp]` chỉ đọc **hàng `i-1`**. Vậy không cần giữ cả bảng: chỉ cần **một mảng** `dp[j]` (độ dài `m + 1`), mỗi lần đọc thêm một ký tự `s[i-1]` thì **cập nhật tại chỗ** mảng đó thành hàng mới.

Khi cập nhật tại chỗ, công thức trở thành:

```text
dp[j] += dp[j-1]     (nếu ký tự hiện tại == t[j-1])
```

- `dp[j]` bên phải (trước khi cộng) đang giữ giá trị **hàng cũ** `dp[i-1][j]` → đúng phần "không dùng".
- `dp[j-1]` phải là **hàng cũ** `dp[i-1][j-1]` → phần "dùng".

Vấn đề nằm ở chữ "phải là hàng cũ". Cùng thử cách tự nhiên nhất: duyệt `j` **xuôi** từ 1 đến `m`.

In [22]:
def num_distinct_1d_forward_wrong(s: str, t: str) -> int:
    m = len(t)
    dp = [1] + [0] * m
    for ch in s:
        for j in range(1, m + 1):          # duyệt XUÔI -> sai
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[m]


for s, t in [("a", "aa"), ("rabbbit", "rabbit"), ("babgbag", "bag")]:
    wrong = num_distinct_1d_forward_wrong(s, t)
    right = num_distinct_dp(s, t)
    print(f"s={s!r:10} t={t!r:9} duyệt xuôi={wrong}  đúng={right}  {'OK' if wrong == right else '<-- SAI'}")

s='a'        t='aa'      duyệt xuôi=1  đúng=0  <-- SAI
s='rabbbit'  t='rabbit'  duyệt xuôi=6  đúng=3  <-- SAI
s='babgbag'  t='bag'     duyệt xuôi=5  đúng=5  OK


### Vì sao duyệt xuôi sai? Một ký tự bị dùng hai lần

Để ý: `"babgbag"`/`"bag"` vẫn ra đúng 5 khi duyệt xuôi! Lý do: `t = "bag"` không có ký tự lặp, nên một ký tự của `s` không thể khớp hai vị trí của `t`. Lỗi chỉ lộ ra khi `t` có ký tự lặp (`"aa"`, `"rabbit"` có hai chữ `b`) → bộ test **phải** có những ca như vậy.

Xét `s = "a"`, `t = "aa"`. Rõ ràng **0** cách (s chỉ có một chữ `a`). Duyệt xuôi:

| Bước | Trước | Phép tính | Sau |
|---|---|---|---|
| bắt đầu | — | — | `dp = [1, 0, 0]` |
| ch='a', j=1 | `dp = [1, 0, 0]` | `dp[1] += dp[0]` = 0 + 1 | `dp = [1, 1, 0]` |
| ch='a', j=2 | `dp = [1, 1, 0]` | `dp[2] += dp[1]` = 0 + **1** | `dp = [1, 1, 1]` ← sai! |

Ở bước j=2, `dp[1]` **vừa được cập nhật** trong chính lượt của ký tự `'a'` này (nó đã là giá trị **hàng mới**). Cộng nó vào nghĩa là: chữ `a` duy nhất của `s` vừa khớp `t[0]` **và** lại khớp luôn `t[1]` — một ký tự bị dùng hai lần.

### Duyệt ngược sửa được vì sao?

Nếu duyệt `j` từ `m` **xuống** 1: khi cập nhật `dp[j]` ta đọc `dp[j-1]` — ô có chỉ số **nhỏ hơn**, mà các ô nhỏ hơn **chưa** được đụng tới trong lượt này → vẫn là giá trị hàng cũ. Đúng như công thức cần.

Quy tắc chung (giống bài **0/1 knapsack**): khi `dp[j]` mới phụ thuộc `dp[j-1]` **cũ**, hãy duyệt `j` **từ lớn xuống nhỏ**; mỗi "vật" (ở đây là mỗi ký tự của `s`) chỉ được dùng tối đa một lần.

In [23]:
def num_distinct_1d(s: str, t: str) -> int:
    m = len(t)
    dp = [1] + [0] * m                     # dp[j]: số cách tạo t[:j] từ phần s đã đọc
    for ch in s:
        for j in range(m, 0, -1):          # duyệt NGƯỢC: dp[j-1] vẫn là giá trị hàng cũ
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[m]


print(num_distinct_1d("a", "aa"))             # 0
print(num_distinct_1d("rabbbit", "rabbit"))   # 3
print(num_distinct_1d("babgbag", "bag"))      # 5

0
3
5


### Trace Cách 5 trên `"rabbbit"` / `"rabbit"`

`trace_1d` in mảng `dp` sau khi đọc mỗi ký tự của `s`, kèm các `j` đã được cộng (theo thứ tự duyệt ngược). So với bảng 2D ở mục 9: **mảng sau ký tự thứ `i` đúng bằng hàng `i` của bảng**.

In [24]:
def trace_1d(s: str, t: str) -> int:
    m = len(t)
    dp = [1] + [0] * m
    print(f"{'':16} dp = {dp}")
    for i, ch in enumerate(s, start=1):
        updated = []
        for j in range(m, 0, -1):
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
                updated.append(j)
        print(f"i={i} ch='{ch}' j={str(updated):6} dp = {dp}")
    return dp[m]


print("Đáp án:", trace_1d("rabbbit", "rabbit"))

                 dp = [1, 0, 0, 0, 0, 0, 0]
i=1 ch='r' j=[1]    dp = [1, 1, 0, 0, 0, 0, 0]
i=2 ch='a' j=[2]    dp = [1, 1, 1, 0, 0, 0, 0]
i=3 ch='b' j=[4, 3] dp = [1, 1, 1, 1, 0, 0, 0]
i=4 ch='b' j=[4, 3] dp = [1, 1, 1, 2, 1, 0, 0]
i=5 ch='b' j=[4, 3] dp = [1, 1, 1, 3, 3, 0, 0]
i=6 ch='i' j=[5]    dp = [1, 1, 1, 3, 3, 3, 0]
i=7 ch='t' j=[6]    dp = [1, 1, 1, 3, 3, 3, 3]
Đáp án: 3


Hãy nhìn dòng `i=4 ch='b' j=[4, 3]`: chữ `b` thứ hai khớp cả `t[2]` và `t[3]`. Duyệt ngược nên `j=4` được tính **trước**: `dp[4] += dp[3]` dùng `dp[3] = 1` (hàng cũ) → `dp[4] = 1`. Sau đó mới `dp[3] += dp[2]` → `dp[3] = 2`. Nếu duyệt xuôi, `dp[3]` sẽ thành 2 trước rồi `dp[4]` nhận nhầm 2.

**Độ phức tạp Cách 5:** Time `O(n·m)` (như Cách 4), **Space `O(m)`** — chỉ một mảng độ dài `m + 1`. Đây là bản gọn và nhanh nhất trong Python, ta chọn làm bản nộp.

## 11. Code nộp LeetCode

Chuyển `num_distinct_1d` vào đúng khung `class Solution` / `def numDistinct(self, s: str, t: str) -> int`. Thêm một dòng kiểm tra nhanh: nếu `t` dài hơn `s` thì trả 0 ngay (không bắt buộc, nhưng rẻ và rõ ý).

In [25]:
class Solution:
    def numDistinct(self, s: str, t: str) -> int:
        m = len(t)
        if m > len(s):
            return 0
        dp = [1] + [0] * m
        for ch in s:
            for j in range(m, 0, -1):
                if ch == t[j - 1]:
                    dp[j] += dp[j - 1]
        return dp[m]


# Bản 2D tương đương (dễ đọc, dễ debug hơn) — cũng được LeetCode chấp nhận.
# Đặt tên khác để không ghi đè Solution ở trên; khi nộp thì đổi lại thành Solution.
class SolutionTable:
    def numDistinct(self, s: str, t: str) -> int:
        n, m = len(s), len(t)
        dp = [[0] * (m + 1) for _ in range(n + 1)]
        for i in range(n + 1):
            dp[i][0] = 1
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                dp[i][j] = dp[i - 1][j]
                if s[i - 1] == t[j - 1]:
                    dp[i][j] += dp[i - 1][j - 1]
        return dp[n][m]

### Bộ test bằng `assert`

Gồm 2 ví dụ của đề và các edge case: `t` dài hơn `s`, `s == t`, một ký tự, toàn chữ giống nhau (đáp án là tổ hợp `C(n, m)`), phân biệt hoa/thường (đề nói "English letters" nên có thể có chữ hoa), và ca bẫy duyệt xuôi `("a", "aa")`.

In [26]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")


sol, sol2 = Solution(), SolutionTable()
tests = [
    ("Ví dụ 1", "rabbbit", "rabbit", 3),
    ("Ví dụ 2", "babgbag", "bag", 5),
    ("t dài hơn s", "abc", "abcd", 0),
    ("s == t", "abc", "abc", 1),
    ("một ký tự khớp", "a", "a", 1),
    ("một ký tự không khớp", "a", "b", 0),
    ("bẫy duyệt xuôi", "a", "aa", 0),
    ("toàn chữ a: C(10, 3)", "a" * 10, "aaa", math.comb(10, 3)),
    ("hoa khác thường", "aAa", "a", 2),
    ("không có ký tự nào khớp", "xyz", "a", 0),
]
for name, s, t, expected in tests:
    check(name, sol.numDistinct(s, t), expected)
    assert sol2.numDistinct(s, t) == expected

PASS  Ví dụ 1: 3
PASS  Ví dụ 2: 5
PASS  t dài hơn s: 0
PASS  s == t: 1
PASS  một ký tự khớp: 1
PASS  một ký tự không khớp: 0
PASS  bẫy duyệt xuôi: 0
PASS  toàn chữ a: C(10, 3): 120
PASS  hoa khác thường: 2
PASS  không có ký tự nào khớp: 0


## 12. Test đối chiếu chéo 5 cách trên input ngẫu nhiên

Brute force (Cách 1) chậm nhưng **đúng theo định nghĩa** → dùng làm trọng tài. Ta sinh ngẫu nhiên `s`, `t` nhỏ trên bảng chữ cái ít ký tự (`"ab"`, `"abc"`) để có nhiều ký tự trùng — nơi dễ sai nhất — và so 5 cách với nhau. `random.seed(115)` cố định để chạy lại cho cùng kết quả.

In [27]:
import random

random.seed(115)
methods = {
    "recursive": num_distinct_recursive,
    "memo": num_distinct_memo,
    "dp2d": num_distinct_dp,
    "dp1d": num_distinct_1d,
    "Solution": Solution().numDistinct,
}
n_tests = 0
for _ in range(400):
    alphabet = random.choice(["ab", "abc"])
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 10)))
    t = "".join(random.choice(alphabet) for _ in range(random.randint(1, 4)))
    expected = num_distinct_bruteforce(s, t)
    for name, fn in methods.items():
        assert fn(s, t) == expected, (name, s, t, fn(s, t), expected)
    n_tests += 1
print(f"Đã so {n_tests} bộ input ngẫu nhiên: cả 5 cách đều khớp brute force.")

Đã so 400 bộ input ngẫu nhiên: cả 5 cách đều khớp brute force.


### Kiểm tra tốc độ với kích thước tối đa

`n = m = 1000` là ca nặng nhất: `10⁶` ô. Ta dùng `s` toàn chữ `a` và `t` gồm 999 chữ `a` (đáp án `C(1000, 999) = 1000`, vừa 32-bit) để chắc chắn ký tự khớp ở mọi ô.

In [28]:
import time

s, t = "a" * 1000, "a" * 999
for name, fn in [("Cách 4 (bảng 2D)", num_distinct_dp), ("Cách 5 (mảng 1D)", Solution().numDistinct)]:
    start = time.perf_counter()
    ans = fn(s, t)
    elapsed = time.perf_counter() - start
    print(f"{name}: đáp án = {ans}, thời gian ~ {elapsed:.2f} giây")

Cách 4 (bảng 2D): đáp án = 1000, thời gian ~ 0.18 giây


Cách 5 (mảng 1D): đáp án = 1000, thời gian ~ 0.07 giây


## 13. Vì sao thuật toán đúng?

**Mệnh đề (invariant) của bảng:** với mọi `0 ≤ i ≤ n`, `0 ≤ j ≤ m`, `dp[i][j]` đúng bằng số tập chỉ số tăng dần trong `{0, ..., i-1}` có `j` phần tử mà đọc ra `t[:j]`.

Chứng minh theo kiểu "quy nạp" bằng lời:

1. **Base case đúng:** `dp[i][0] = 1` vì chỉ có tập rỗng đọc ra chuỗi rỗng. `dp[0][j] = 0` (j ≥ 1) vì không có chỉ số nào để chọn.
2. **Bước chuyển đúng:** giả sử hàng `i-1` đã đúng. Xét mọi cách chọn hợp lệ cho `(s[:i], t[:j])`. Chia chúng thành **hai nhóm rời nhau** theo câu hỏi "chỉ số `i-1` có được chọn không?":
   - **Không chọn `i-1`:** đó chính xác là các cách chọn của `(s[:i-1], t[:j])` → có `dp[i-1][j]` cách.
   - **Chọn `i-1`:** vì chỉ số tăng dần, `i-1` là chỉ số **cuối cùng** được chọn nên nó phải khớp ký tự cuối `t[j-1]`. Nếu `s[i-1] != t[j-1]` thì nhóm này rỗng. Nếu khớp, phần còn lại là một cách chọn của `(s[:i-1], t[:j-1])` → có `dp[i-1][j-1]` cách.
   - Hai nhóm **không trùng nhau** (khác nhau ở chỉ số `i-1`) và **phủ hết** mọi cách → cộng lại là đúng, không đếm thiếu, không đếm thừa.
3. Điền hàng theo thứ tự `i` tăng dần nên khi tính hàng `i`, hàng `i-1` đã đúng. Kết luận `dp[n][m]` là đáp án.

**Cách 5 đúng** vì nó tính **cùng** các giá trị đó: duyệt `j` ngược đảm bảo khi cập nhật `dp[j]`, cả `dp[j]` (chưa sửa) lẫn `dp[j-1]` (chưa sửa, vì chỉ số nhỏ hơn được sửa sau) đều còn là giá trị hàng `i-1`. Nên sau lượt ký tự thứ `i`, mảng bằng đúng hàng `i` của bảng (trace mục 10 đã cho thấy).

**Cách 2, 3 đúng** vì cùng lập luận "chia hai nhóm rời nhau theo `s[i]` có được chọn không", chỉ là nhìn từ hậu tố.

## 14. So sánh các cách

Ký hiệu `n = len(s)`, `m = len(t)` (đều ≤ 1000).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử mọi bộ chỉ số `combinations` | `O(C(n,m)·m)` | `O(m)` | chỉ dùng làm trọng tài cho input nhỏ |
| 2. Đệ quy thuần | chọn / không chọn `s[i]` | mũ (exponential) | `O(n)` stack | tính lại bài toán con trùng |
| 3. Đệ quy + memo | nhớ `f(i, j)` | `O(n·m)` | `O(n·m)` + `O(n)` stack | bẫy `RecursionError` với n = 1000 |
| 4. DP bảng 2D | điền `dp[i][j]` từ trên xuống | `O(n·m)` | `O(n·m)` | dễ in, dễ debug, dễ trace |
| 5. DP 1D | một mảng, duyệt `j` ngược | `O(n·m)` | `O(m)` | **bản nộp**: gọn và nhanh nhất |

Lộ trình tư duy: **đếm từng cách** (1) → **chia thành bài toán con** (2) → **nhớ bài toán con** (3) → **điền theo thứ tự, bỏ đệ quy** (4) → **chỉ giữ hàng cần thiết** (5).

## 15. Các lỗi người mới hay gặp

**Lỗi 1 — Hiểu "distinct" là đếm chuỗi khác nhau.** Dùng `set` thì đáp án luôn ≤ 1. Đề đếm **cách chọn chỉ số**.

**Lỗi 2 — Khi khớp thì chỉ "dùng", quên lựa chọn "bỏ qua".** Viết `dp[i][j] = dp[i-1][j-1]` khi khớp (thay vì `dp[i-1][j] + dp[i-1][j-1]`). Với `"rabbbit"` sẽ mất các cách mà một chữ `b` khớp được nhưng ta cố tình bỏ nó để dùng chữ `b` sau.

**Lỗi 3 — Quên hạt giống `dp[i][0] = 1`** (chỉ đặt `dp[0][0] = 1`). Khi đó chữ cái đầu của `t` chỉ khớp được với `s[0]`, mọi cách bắt đầu ở vị trí sau đều bị mất.

**Lỗi 4 — Duyệt `j` xuôi trong DP 1D** (đã phân tích ở mục 10): một ký tự bị dùng cho hai vị trí của `t`.

**Lỗi 5 — Base case đệ quy sai thứ tự:** kiểm tra `i == len(s)` trước `j == len(t)` → ca "cả hai cùng hết" trả 0 thay vì 1.

**Lỗi 6 — Tạo bảng bằng `[[0]*(m+1)]*(n+1)`** (mục 8) và **quên tăng recursion limit** cho bản memo (mục 7).

Cell dưới chạy các phiên bản sai của lỗi 2, 3, 5 để thấy chúng thất bại ở đâu.

In [29]:
def wrong_only_take(s, t):            # Lỗi 2
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = 1
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]          # SAI: thiếu + dp[i-1][j]
            else:
                dp[i][j] = dp[i - 1][j]
    return dp[n][m]


def wrong_seed(s, t):                 # Lỗi 3
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    dp[0][0] = 1                                     # SAI: phải là dp[i][0] = 1 với mọi i
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = dp[i - 1][j] + (dp[i - 1][j - 1] if s[i - 1] == t[j - 1] else 0)
    return dp[n][m]


def wrong_base_order(s, t):           # Lỗi 5
    def f(i, j):
        if i == len(s):                              # SAI: kiểm tra s hết trước
            return 0
        if j == len(t):
            return 1
        return f(i + 1, j) + (f(i + 1, j + 1) if s[i] == t[j] else 0)
    return f(0, 0)


for s, t in [("rabbbit", "rabbit"), ("babgbag", "bag"), ("abc", "abc")]:
    print(f"{s!r:10} {t!r:9} đúng={num_distinct_1d(s, t)} | chỉ-dùng={wrong_only_take(s, t)} "
          f"| thiếu-hạt-giống={wrong_seed(s, t)} | base-sai-thứ-tự={wrong_base_order(s, t)}")

'rabbbit'  'rabbit'  đúng=3 | chỉ-dùng=1 | thiếu-hạt-giống=3 | base-sai-thứ-tự=0
'babgbag'  'bag'     đúng=5 | chỉ-dùng=1 | thiếu-hạt-giống=3 | base-sai-thứ-tự=1
'abc'      'abc'     đúng=1 | chỉ-dùng=1 | thiếu-hạt-giống=1 | base-sai-thứ-tự=0


Đọc kết quả:

- **chỉ-dùng**: `"rabbbit"` ra 1 thay vì 3 — chỉ đếm được cách "tham lam" dùng ngay chữ `b` khớp được.
- **thiếu-hạt-giống**: `"babgbag"` ra 3 thay vì 5 — chỉ đếm được 3 cách mà chữ `b` đầu tiên của `t` khớp với `s[0]`; mất 2 cách dùng chữ `b` ở chỉ số 2 và 4. `"rabbbit"` vẫn ra 3 một cách "may mắn" vì chữ `r` duy nhất nằm ở đầu `s` → lỗi này **không lộ** với Ví dụ 1, phải có Ví dụ 2 mới bắt được. Bài học: luôn test nhiều ca.
- **base-sai-thứ-tự**: `"abc"`/`"abc"` ra 0 thay vì 1 — khi chữ cuối của `s` được dùng, ta tới `f(n, m)` và base case sai trả 0.

Cách sửa: dùng đúng công thức "trên + chéo", đặt cả cột 0 bằng 1, kiểm tra `j == len(t)` trước.

## 16. Mẹo nhận dạng pattern

Khi gặp đề có các dấu hiệu sau, hãy nghĩ ngay tới **DP 2 chiều trên hai chuỗi** `dp[i][j]` (tiền tố `s[:i]` và `t[:j]`):

- Có **hai chuỗi** (hoặc hai dãy) và câu hỏi về **subsequence**, khớp, biến đổi chuỗi này thành chuỗi kia.
- Đề hỏi **"bao nhiêu cách"** (đếm → dùng `+`), **"dài nhất / ít nhất"** (tối ưu → dùng `max`/`min`), hoặc **"có được không"** (→ dùng `or`).
- Mỗi bước có **vài lựa chọn** cho ký tự cuối: dùng / bỏ (subsequence), khớp / chèn / xóa / thay (edit distance)...

Công thức suy nghĩ 4 bước dùng được cho cả họ bài này:

1. **Định nghĩa trạng thái** bằng một câu tiếng Việt rõ ràng ("`dp[i][j]` là số cách ... của `s[:i]` và `t[:j]`").
2. **Chia trường hợp** theo ký tự cuối, đảm bảo các trường hợp **rời nhau và phủ hết** (với bài đếm).
3. **Base case** cho hàng 0 và cột 0 — hỏi "chuỗi rỗng thì sao?".
4. **Thứ tự điền** + có thể **nén còn 1 chiều** (chú ý hướng duyệt).

Dấu hiệu cần **duyệt ngược khi nén 1D**: ô mới cần ô **chỉ số nhỏ hơn** của **hàng cũ** (như bài này, 0/1 knapsack). Nếu cần ô chỉ số nhỏ hơn của **hàng mới** (như unbounded knapsack / coin change đếm cách) thì duyệt xuôi.

## 17. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 392 — Is Subsequence | chỉ hỏi có / không → hai con trỏ (mục 2) |
| LeetCode 1143 — Longest Common Subsequence | cùng bảng `dp[i][j]` trên hai tiền tố, dùng `max` thay `+` |
| LeetCode 583 — Delete Operation for Two Strings | dựa trên LCS |
| LeetCode 72 — Edit Distance | `dp[i][j]` với 3 thao tác, dùng `min` |
| LeetCode 940 — Distinct Subsequences II | đếm subsequence **khác nhau về chuỗi** của một chuỗi — đúng nghĩa "distinct" mà bài 115 *không* dùng (Bài tập 6) |
| LeetCode 10 / 44 — Regex / Wildcard Matching | DP 2 chiều trên chuỗi và pattern, dùng `or` |

**Tối ưu thêm cho Cách 5 (tham khảo):** với mỗi ký tự `ch` của `s`, thay vì duyệt cả `m` vị trí của `t`, ta chỉ duyệt các vị trí `j` mà `t[j-1] == ch` (tính sẵn bằng `dict`, sắp **giảm dần** để giữ đúng thứ tự duyệt ngược). Khi bảng chữ cái lớn, số phép tính giảm mạnh; độ phức tạp xấu nhất vẫn `O(n·m)`.

In [30]:
from collections import defaultdict


def num_distinct_1d_positions(s: str, t: str) -> int:
    positions = defaultdict(list)            # ký tự -> các j (1-based) mà t[j-1] == ký tự
    for j in range(len(t), 0, -1):           # thêm j giảm dần -> list đã sắp giảm dần
        positions[t[j - 1]].append(j)
    dp = [1] + [0] * len(t)
    for ch in s:
        for j in positions.get(ch, []):      # chỉ các j khớp, theo thứ tự ngược
            dp[j] += dp[j - 1]
    return dp[len(t)]


t_demo = "rabbit"
pos_demo = defaultdict(list)
for j in range(len(t_demo), 0, -1):
    pos_demo[t_demo[j - 1]].append(j)
print("Vị trí (1-based, giảm dần) trong t='rabbit':", dict(pos_demo))
print(num_distinct_1d_positions("rabbbit", "rabbit"), num_distinct_1d_positions("babgbag", "bag"))

Vị trí (1-based, giảm dần) trong t='rabbit': {'t': [6], 'i': [5], 'b': [4, 3], 'a': [2], 'r': [1]}
3 5


## 18. Bài tập vận dụng

Làm theo thứ tự, từ dễ đến khó. **Đừng xem đáp án ở cuối notebook trước khi tự làm.** Các cell TODO chỉ định nghĩa hàm, test đang được comment — bạn làm xong thì bỏ comment để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

Đoán `numDistinct(s, t)` cho từng cặp, giải thích bằng cách đếm tổ hợp:

1. `s = "aaa"`, `t = "a"`
2. `s = "aaa"`, `t = "aa"`
3. `s = "bbb"`, `t = "bbb"`
4. `s = "abc"`, `t = "abcd"`
5. `s = "ababc"`, `t = "abc"`
6. `s = "abcabc"`, `t = "abc"`

### Bài 2 — Tính tay bảng DP

Lập bảng `dp[i][j]` (8 hàng × 4 cột) cho Ví dụ 2: `s = "babgbag"`, `t = "bag"`. Với mỗi ô khớp, ghi rõ "trên + chéo". Kiểm tra góc dưới phải bằng 5. Sau đó so với `print_dp_table("babgbag", "bag")`.

### Bài 3 — Điền TODO: đệ quy + memo

Hoàn thiện Cách 3 theo các TODO. Nhớ thứ tự base case.

In [31]:
from functools import lru_cache


def exercise_num_distinct_memo(s: str, t: str) -> int:
    n, m = len(s), len(t)

    @lru_cache(maxsize=None)
    def f(i: int, j: int) -> int:
        # TODO 1: nếu đã khớp hết t -> trả 1
        # TODO 2: nếu s cạn (hoặc không đủ ký tự) -> trả 0
        # TODO 3: ways = số cách khi bỏ qua s[i]
        # TODO 4: nếu s[i] == t[j], cộng thêm số cách khi dùng s[i]
        raise NotImplementedError

    return f(0, 0)


# Tự làm xong rồi mới mở test:
# assert exercise_num_distinct_memo("rabbbit", "rabbit") == 3
# assert exercise_num_distinct_memo("babgbag", "bag") == 5
# assert exercise_num_distinct_memo("abc", "abc") == 1

### Bài 4 — Tự viết lại DP 1 chiều

Không nhìn mục 10, tự viết lại bản 1D. Tự kiểm tra bằng ca bẫy `("a", "aa")` — nếu ra 1 thì bạn đang duyệt xuôi.

In [32]:
def exercise_num_distinct_1d(s: str, t: str) -> int:
    # TODO: tạo mảng dp độ dài len(t) + 1, dp[0] = 1
    # TODO: với mỗi ký tự ch của s, duyệt j theo chiều đúng và cập nhật dp[j]
    pass


# assert exercise_num_distinct_1d("a", "aa") == 0
# assert exercise_num_distinct_1d("rabbbit", "rabbit") == 3
# assert exercise_num_distinct_1d("babgbag", "bag") == 5

### Bài 5 — Biến thể: liệt kê mọi cách chọn

Viết `exercise_list_ways(s, t)` trả về **danh sách mọi bộ chỉ số** (tuple tăng dần) đọc ra `t`, theo thứ tự từ điển. Yêu cầu: **không** dùng `combinations` thử hết; hãy dùng đệ quy chọn / không chọn và **cắt nhánh** nhờ bảng đếm (nhánh nào có 0 cách thì khỏi đi vào). Số phần tử của kết quả phải bằng `numDistinct(s, t)`.

Ví dụ: `exercise_list_ways("babgbag", "bag")` → `[(0, 1, 3), (0, 1, 6), (0, 5, 6), (2, 5, 6), (4, 5, 6)]`.

In [33]:
def exercise_list_ways(s: str, t: str) -> list:
    # TODO 1: tính bảng đếm theo HẬU TỐ: cnt[i][j] = số cách s[i:] tạo t[j:]
    # TODO 2: đệ quy go(i, j, chosen): nếu j == len(t) -> lưu tuple(chosen)
    # TODO 3: nhánh DÙNG s[i] (nếu khớp và cnt[i+1][j+1] > 0) đi trước để có thứ tự từ điển,
    #         rồi nhánh BỎ QUA s[i] (nếu cnt[i+1][j] > 0)
    pass


# assert exercise_list_ways("babgbag", "bag") == [(0, 1, 3), (0, 1, 6), (0, 5, 6), (2, 5, 6), (4, 5, 6)]
# assert len(exercise_list_ways("rabbbit", "rabbit")) == 3

### Bài 6 — Mini challenge: Distinct Subsequences II (LeetCode 940)

Đề (LeetCode 940): cho chuỗi `s`, trả về số **subsequence khác rỗng, khác nhau về chuỗi** của `s`. Vì đáp án có thể rất lớn, trả về kết quả **modulo `10^9 + 7`**. Ràng buộc: `1 <= s.length <= 2000`, `s` gồm chữ cái thường.

Ví dụ: `"abc"` → 7 (`a, b, c, ab, ac, bc, abc`); `"aba"` → 6 (`a, b, ab, aa, ba, aba`); `"aaa"` → 3 (`a, aa, aaa`).

Lần này "distinct" **đúng nghĩa là chuỗi khác nhau** — đúng cái mà bài 115 không phải! Viết lời giải `O(n)` (hoặc `O(26·n)`), signature LeetCode là `def distinctSubseqII(self, s: str) -> int`.

In [34]:
def exercise_distinct_subseq_ii(s: str) -> int:
    MOD = 10**9 + 7
    # TODO: ends[c] = số subsequence khác nhau (khác rỗng) KẾT THÚC bằng ký tự c
    # TODO: khi đọc ký tự c: mọi subsequence hiện có (kể cả rỗng) nối thêm c
    #       -> ends[c] = (tổng mọi ends + 1) % MOD   (ghi đè, không cộng dồn: tránh đếm trùng)
    # TODO: đáp án = tổng mọi ends % MOD
    pass


# assert exercise_distinct_subseq_ii("abc") == 7
# assert exercise_distinct_subseq_ii("aba") == 6
# assert exercise_distinct_subseq_ii("aaa") == 3

## 19. Gợi ý theo mức

**Bài 1**
- Mức 1: chuỗi toàn một chữ → đáp án là `C(len(s), len(t))`.
- Mức 2: với `"ababc"`, chữ `c` chỉ có một → chọn một cặp `a` trước `b` trong `"abab"`.
- Mức 3: liệt kê vị trí: `a` ở {0, 2}, `b` ở {1, 3}; đếm các cặp `a < b`.

**Bài 2**
- Mức 1: cột 0 toàn 1, hàng 0 còn lại toàn 0.
- Mức 2: ô không khớp = chép ô ngay trên; ô khớp = trên + chéo trên-trái.
- Mức 3: hàng `i=6` (`a`): `dp[6][2] = dp[5][2] + dp[5][1] = 1 + 3`.

**Bài 3**
- Mức 1: giống hệt Cách 2, chỉ thêm decorator (đã có sẵn).
- Mức 2: thứ tự: `j == m` trước, rồi `n - i < m - j`.
- Mức 3: `ways = f(i + 1, j)`; `if s[i] == t[j]: ways += f(i + 1, j + 1)`.

**Bài 4**
- Mức 1: `dp = [1] + [0] * m`.
- Mức 2: `for j in range(m, 0, -1)`.
- Mức 3: `if ch == t[j - 1]: dp[j] += dp[j - 1]`.

**Bài 5**
- Mức 1: `cnt[i][len(t)] = 1` với mọi `i`; `cnt[i][j] = cnt[i+1][j] + (cnt[i+1][j+1] nếu s[i]==t[j])`, điền `i` giảm dần.
- Mức 2: trong `go`, dùng một list `chosen` chung: `append` trước khi đi nhánh dùng, `pop` sau khi về (backtracking).
- Mức 3: điều kiện vào nhánh dùng: `s[i] == t[j] and cnt[i+1][j+1] > 0`; nhánh bỏ: `cnt[i+1][j] > 0`. Nhờ cắt nhánh, mọi nhánh đi vào đều dẫn tới ít nhất một kết quả.

**Bài 6**
- Mức 1: phân loại subsequence theo **ký tự cuối**.
- Mức 2: khi gặp `c` lần nữa, mọi subsequence kết thúc bằng `c` trước đó đều nằm lại trong tập "(mọi subsequence cũ) + c" → nên **ghi đè** `ends[c]`, không cộng.
- Mức 3: `ends[c] = (sum(ends.values()) + 1) % MOD` (số `+1` là subsequence chỉ gồm `c`, tức nối `c` vào chuỗi rỗng).

## 20. Đáp án tham khảo

### Đáp án Bài 1

| Cặp | Đáp án | Lý do |
|---|---|---|
| `"aaa"`, `"a"` | 3 | `C(3,1)` |
| `"aaa"`, `"aa"` | 3 | `C(3,2)` |
| `"bbb"`, `"bbb"` | 1 | `C(3,3)` |
| `"abc"`, `"abcd"` | 0 | `t` dài hơn `s` |
| `"ababc"`, `"abc"` | 3 | cặp (a, b): (0,1), (0,3), (2,3); `c` ở 4 |
| `"abcabc"`, `"abc"` | 4 | (0,1,2), (0,1,5), (0,4,5), (3,4,5) |


### Đáp án Bài 2 — bảng `s = "babgbag"`, `t = "bag"`

| i \ cột | `-` | `b` | `a` | `g` | Ô khớp (trên + chéo) |
|---|---|---|---|---|---|
| 0 `-` | 1 | 0 | 0 | 0 | |
| 1 `b` | 1 | 1 | 0 | 0 | `dp[1][1] = 0 + 1` |
| 2 `a` | 1 | 1 | 1 | 0 | `dp[2][2] = 0 + 1` |
| 3 `b` | 1 | 2 | 1 | 0 | `dp[3][1] = 1 + 1` |
| 4 `g` | 1 | 2 | 1 | 1 | `dp[4][3] = 0 + 1` |
| 5 `b` | 1 | 3 | 1 | 1 | `dp[5][1] = 2 + 1` |
| 6 `a` | 1 | 3 | 4 | 1 | `dp[6][2] = 1 + 3` |
| 7 `g` | 1 | 3 | 4 | 5 | `dp[7][3] = 1 + 4` |

Đáp án `dp[7][3] = 5`. Ý nghĩa ô `dp[6][2] = 4`: trong `"babgba"` có 4 cách chọn `"ba"` (3 chữ `b` trước chữ `a` cuối, cộng 1 cách dùng chữ `a` đầu).

Cell dưới kiểm chứng cả hai bằng code.

In [35]:
quiz = [("aaa", "a", 3), ("aaa", "aa", 3), ("bbb", "bbb", 1),
        ("abc", "abcd", 0), ("ababc", "abc", 3), ("abcabc", "abc", 4)]
for s, t, expected in quiz:
    assert Solution().numDistinct(s, t) == expected == num_distinct_bruteforce(s, t)
    print(f"{s!r:9} {t!r:7} -> {expected}")

print()
print_dp_table("babgbag", "bag")
assert build_dp_table("babgbag", "bag")[6][2] == 4

'aaa'     'a'     -> 3
'aaa'     'aa'    -> 3
'bbb'     'bbb'   -> 1
'abc'     'abcd'  -> 0
'ababc'   'abc'   -> 3
'abcabc'  'abc'   -> 4

         -   b   a   g
i=0  -  1   0   0   0
i=1  b  1   1   0   0
i=2  a  1   1   1   0
i=3  b  1   2   1   0
i=4  g  1   2   1   1
i=5  b  1   3   1   1
i=6  a  1   3   4   1
i=7  g  1   3   4   5


### Đáp án Bài 3 và Bài 4

In [36]:
from functools import lru_cache


def answer_num_distinct_memo(s: str, t: str) -> int:
    n, m = len(s), len(t)

    @lru_cache(maxsize=None)
    def f(i: int, j: int) -> int:
        if j == m:
            return 1
        if n - i < m - j:
            return 0
        ways = f(i + 1, j)
        if s[i] == t[j]:
            ways += f(i + 1, j + 1)
        return ways

    return f(0, 0)


def answer_num_distinct_1d(s: str, t: str) -> int:
    m = len(t)
    dp = [1] + [0] * m
    for ch in s:
        for j in range(m, 0, -1):
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[m]


for fn in (answer_num_distinct_memo, answer_num_distinct_1d):
    assert fn("rabbbit", "rabbit") == 3
    assert fn("babgbag", "bag") == 5
    assert fn("abc", "abc") == 1
    assert fn("a", "aa") == 0
print("Đáp án Bài 3, Bài 4: PASS")

Đáp án Bài 3, Bài 4: PASS


### Đáp án Bài 5 — liệt kê mọi cách chọn

Bảng `cnt` theo hậu tố chính là Cách 3 viết bottom-up. Nhờ nó, mỗi nhánh đệ quy ta đi vào **chắc chắn** dẫn tới ít nhất một kết quả, nên thời gian chỉ tỉ lệ với (số kết quả × độ dài `s`) chứ không phải `C(n, m)`.

In [37]:
def answer_list_ways(s: str, t: str) -> list:
    n, m = len(s), len(t)
    cnt = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        cnt[i][m] = 1
    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            cnt[i][j] = cnt[i + 1][j] + (cnt[i + 1][j + 1] if s[i] == t[j] else 0)

    result, chosen = [], []

    def go(i: int, j: int) -> None:
        if j == m:
            result.append(tuple(chosen))
            return
        if s[i] == t[j] and cnt[i + 1][j + 1] > 0:   # nhánh dùng s[i] trước -> thứ tự từ điển
            chosen.append(i)
            go(i + 1, j + 1)
            chosen.pop()
        if cnt[i + 1][j] > 0:                         # nhánh bỏ qua s[i]
            go(i + 1, j)

    if cnt[0][0] > 0:
        go(0, 0)
    return result


print(answer_list_ways("babgbag", "bag"))
assert answer_list_ways("babgbag", "bag") == [(0, 1, 3), (0, 1, 6), (0, 5, 6), (2, 5, 6), (4, 5, 6)]
assert answer_list_ways("rabbbit", "rabbit") == ways_example1          # khớp 3 cách ở mục 1
assert answer_list_ways("abc", "abcd") == []

random.seed(940)
for _ in range(200):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 9)))
    t = "".join(random.choice("ab") for _ in range(random.randint(1, 3)))
    brute = [c for c in combinations(range(len(s)), len(t)) if "".join(s[i] for i in c) == t]
    assert answer_list_ways(s, t) == brute
print("Đáp án Bài 5: PASS (khớp brute force trên 200 input ngẫu nhiên)")

[(0, 1, 3), (0, 1, 6), (0, 5, 6), (2, 5, 6), (4, 5, 6)]
Đáp án Bài 5: PASS (khớp brute force trên 200 input ngẫu nhiên)


### Đáp án Bài 6 — Distinct Subsequences II (LeetCode 940)

`ends[c]` = số subsequence **khác nhau** kết thúc bằng `c`. Khi đọc thêm một chữ `c`, tập mọi subsequence kết thúc bằng `c` (sau bước này) đúng bằng: lấy **mọi** subsequence khác nhau hiện có (kể cả rỗng) rồi nối thêm `c`. Tập mới này **chứa trọn** các subsequence kết thúc bằng `c` cũ, nên ta **ghi đè** chứ không cộng dồn — đó là cách loại trùng.

Time `O(26·n)` (hoặc `O(n)` nếu giữ sẵn tổng), Space `O(26)`.

In [38]:
class Solution940:
    def distinctSubseqII(self, s: str) -> int:
        MOD = 10**9 + 7
        ends = {}
        for c in s:
            ends[c] = (sum(ends.values()) + 1) % MOD
        return sum(ends.values()) % MOD


def brute_distinct_subseq_ii(s: str) -> int:
    seen = set()
    for mask in range(1, 2 ** len(s)):
        seen.add("".join(s[i] for i in range(len(s)) if mask >> i & 1))
    return len(seen)


sol940 = Solution940()
assert sol940.distinctSubseqII("abc") == 7
assert sol940.distinctSubseqII("aba") == 6
assert sol940.distinctSubseqII("aaa") == 3
random.seed(7)
for _ in range(200):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 10)))
    assert sol940.distinctSubseqII(s) == brute_distinct_subseq_ii(s)
print("Đáp án Bài 6: PASS | 'babgbag' có", sol940.distinctSubseqII("babgbag"), "subsequence khác nhau")

Đáp án Bài 6: PASS | 'babgbag' có 84 subsequence khác nhau


## 21. Template ngắn nên nhớ

```python
# Đếm số cách t là subsequence của s  —  O(n·m) time, O(m) space
dp = [1] + [0] * len(t)          # dp[j]: số cách tạo t[:j]; t rỗng có 1 cách
for ch in s:
    for j in range(len(t), 0, -1):   # NGƯỢC: mỗi ký tự của s dùng tối đa 1 lần
        if ch == t[j - 1]:
            dp[j] += dp[j - 1]       # cũ(bỏ ch) + cũ(dùng ch khớp t[j-1])
return dp[len(t)]
```

Bản 2D để suy nghĩ:

```text
dp[i][0] = 1 ;  dp[0][j>0] = 0
dp[i][j] = dp[i-1][j] + (dp[i-1][j-1] nếu s[i-1] == t[j-1])
```

Câu thần chú: **"Mỗi ký tự của `s`: bỏ nó — hoặc, nếu khớp, dùng nó. Đếm thì CỘNG hai nhánh. Nén 1D thì duyệt NGƯỢC."**

## 22. Checklist tự kiểm tra trước khi submit

- Mình có đếm **cách chọn chỉ số** (không dùng `set` loại chuỗi trùng)?
- Khi ký tự khớp, có cộng **cả hai** nhánh (bỏ + dùng), không chỉ nhánh dùng?
- Base case: cột `t` rỗng bằng **1** cho **mọi** `i` (bản 1D: `dp[0] = 1`)? Bản đệ quy: kiểm tra `j == len(t)` **trước** `i == len(s)`?
- Bản 1D: vòng `j` có chạy **`range(m, 0, -1)`** không? Đã thử ca bẫy `("a", "aa") → 0`?
- Chỉ số lệch 1: ô `dp[i][j]` so `s[i-1]` với `t[j-1]`?
- Bảng 2D tạo bằng list comprehension, không dùng `[[0]*(m+1)]*(n+1)`?
- Bản memo: đã tính tới độ sâu đệ quy `n + 1` (tăng `sys.setrecursionlimit` hoặc dùng bottom-up)?
- Đã test 2 ví dụ đề + `t` dài hơn `s` + `s == t` + chuỗi toàn một chữ (so với `C(n, m)`)?
- (Nếu viết bằng C++/Java) đã lo tràn số ở giá trị trung gian chưa?